<a href="https://colab.research.google.com/github/Imran1hp/mini-gpt-transformer/blob/main/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
!wget https://raw.githubusercontent.com/Imran1hp/mini-gpt-transformer/refs/heads/main/input.txt

--2026-08-23 17:26:44--  https://raw.githubusercontent.com/Imran1hp/mini-gpt-transformer/refs/heads/main/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-08-23 17:26:44 (36.7 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [72]:
with open('input.txt' ,'r' ,encoding='utf-8') as f:
  text =f.read()

In [73]:
len(text)

1115394

In [74]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [75]:
chars = sorted(list(set(text)))
print(''.join(chars))
print('Lenght of the character',len(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Lenght of the character 65


In [76]:
stoi = {ch:i for i , ch in enumerate(chars)}
iots = {i:ch for i , ch in enumerate(chars)}
encode = lambda s: [stoi[c]for c in s]
decode = lambda l :''.join([iots[n] for n in l])


In [77]:
import torch

data = torch.tensor(encode(text) ,dtype = torch.long)
print(data.shape , data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [78]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]


In [79]:
print(f"train len:{len(train_data)}")
print(f"test len: {len(val_data)}")

train len:1003854
test len: 111540


In [80]:
torch.manual_seed(1337)
BATCH_SIZE =4
BLOCK_SIZE =8

def get_batch(split , batch_size , block_size):
  data = train_data if split =='train'else val_data
  ix = torch.randint(len(data) - block_size , (batch_size,))
  x = torch.stack([data[i: i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x , y

xb ,yb= get_batch('train' , 4 , 8)

print('inputs:')
print(xb.shape)
print(xb)

print('Output')
print(yb.shape)
print(yb)

print("--------------------------------------------------------")
for b in range(BATCH_SIZE):
  for t in range(BLOCK_SIZE):
    context= xb[b ,:t+1]
    target =yb[b ,t]

    print(f"When input is {context.tolist()} the targte: {target.tolist()} ")


inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Output
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
--------------------------------------------------------
When input is [24] the targte: 43 
When input is [24, 43] the targte: 58 
When input is [24, 43, 58] the targte: 5 
When input is [24, 43, 58, 5] the targte: 57 
When input is [24, 43, 58, 5, 57] the targte: 1 
When input is [24, 43, 58, 5, 57, 1] the targte: 46 
When input is [24, 43, 58, 5, 57, 1, 46] the targte: 43 
When input is [24, 43, 58, 5, 57, 1, 46, 43] the targte: 39 
When input is [44] the targte: 53 
When input is [44, 53] the targte: 56 
When input is [44, 53, 56] the targte: 1 
When input is [44, 53, 56, 1] the targte: 58 
When inp

In [81]:
ix= torch.randint(len(train_data) - BLOCK_SIZE , (BATCH_SIZE,))
ix

tensor([971401, 579495, 193625, 348340])

# Bigramlanguage  model

In [82]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
vocab_size = 65

class BigramLanguageModel(nn.Module):

  def __init__(self ,vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size , vocab_size)

  def forward( self ,inputs ,targets =None):

    logits = self.token_embedding_table(inputs) # (B=batch[4], T= token/block_size[8], C= channels[65] )

    if targets is None:
     loss =None

    else:
      # Reshape logits and targets for F.cross_entropy
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits , targets)

    return logits , loss

  def generate(self , idx , max_new_token): # there idx is shape is (4,8) batch=4 , block/token=8
    for _ in range(max_new_token):

     logits , loss = self(idx) # doing forword pass
     logits = logits[: ,-1 ,:]  # logits initailly is (4,8,65) we are taking all batch(4) last token/letter and its all 65 next charecters possiblity now shape is(4 ,65)
     probs = F.softmax(logits ,dim=-1) #(4,65)
     idx_next = torch.multinomial(probs , num_samples=1) #(4,1)
     idx = torch.cat((idx , idx_next) , dim =1) #(4 ,9)

    return idx


model_1 = BigramLanguageModel(vocab_size)
logits , loss= model_1(xb , yb)


print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [83]:
idx = torch.zeros((1,1) , dtype =torch.long)
print(decode(model_1.generate(idx , max_new_token=100)[0].tolist()))


SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


# Traning the model_1

In [84]:
optimizer = torch.optim.AdamW(params = model_1.parameters(), lr =1e-3)

In [85]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [86]:

model_1 = model_1.to(device)
for i in range(1000):

  xb , yb = get_batch(split = 'train', block_size = 8 , batch_size = 32)
  xb , yb =xb.to(device) ,yb.to(device)
  logits , loss = model_1( xb , yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

print(loss.item())


3.704136848449707


In [87]:
idx = torch.zeros((1,1), dtype = torch.long)
idx = idx.to(device)
print(decode(model_1.generate(idx , max_new_token= 300)[0].tolist()))


Wh;;Sq.f ustNzknc
kwgOj$dhPWr,SV?hsusiKpgXXUh;Apmem d?hESXI.i;TrJgkiF-oKbXCAA -botrngFCHAUQkn$

pn$w-gHoi?wtd!
LLULIfSK'bAw :M.ZtOptXEQcL?hfaofqbPd?OnonQQJMap$aypupIBYGUsZaI'ottllo..k$W$Akp?yl?ajKlzY!lx&QQLW? t,bXFkyhl-dmVsHeckhRl,jSClgjuk:3Iv
?OqlrV;!Plxfzgy;;
'mRjuBQ&xk!$
h
SiruDJgKuDny,S$ERf.?GSV


#Mathematic trick in self-Attention

In [88]:
B, T , C = 4 , 8 , 2
x = torch.randn(B , T ,C)

In [89]:
#Version 1 one of making the weight matrix
wei = torch.tril( torch.ones(T ,T))
wei = wei / wei.sum( 1 ,keepdim = True)

xbow = wei @ x

In [90]:
#Version 2
tril = torch.tril(torch.ones(T ,T))
wei = torch.zeros( T , T)
wei = wei.masked_fill( tril == 0 ,float('-inf'))
wei = F.softmax( wei , dim=1)
out = wei @ x
out.shape


torch.Size([4, 8, 2])

In [91]:
#Version 4 self_atttention
torch.manual_seed(1337)
B,T,C = 4,8,32

x = torch.rand(B,T,C)

head_size = 16
key = nn.Linear(C,head_size ,bias = False)
query = nn.Linear(C , head_size , bias =False)
value = nn.Linear( C , head_size , bias = False)
k = key(x) #[B , T , 16]
q =query(x) #[ B , T , 16]
v = value(x) # [B ,T  , 16]

wei = q @ k.transpose(-2 , -1) # q -> [B, T, 16] k ->[B ,16 ,T] wei->[ T , T]

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill( tril ==0 , float('-inf'))
wei = F.softmax(wei , dim =1)
out   = wei @ v #  wei ->[ T ,T]    v-> [B, T, 16] and out-> [ B , T , 16]
out.shape


torch.Size([4, 8, 16])

 ## Model_2 with adding positioning embedding and token embedding and attention head

In [92]:
# Single head attention
import torch
import torch.nn as nn
torch.manual_seed(1337)

class Head(nn.Module):
  def __init__(self ,head_size):
    super().__init__()
    self.key= nn.Linear(n_embd ,head_size ,bias = False)
    self.query = nn.Linear(n_embd , head_size , bias =False)
    self.value = nn.Linear(n_embd , head_size , bias = False)
    self.register_buffer('tril',torch.tril(torch.ones(BLOCK_SIZE , BLOCK_SIZE)))

  def forward(self , x):
    B,T,C = x.shape

    k = self.key(x)
    q = self.query(x)
    v = self.value(x);

    wei = q @ k.transpose(-2 ,-1) *C**-0.5
    wei  = wei.masked_fill(self.tril[:T , :T] == 0 , float('-inf'))
    wei = F.softmax(wei ,dim=-1)

    out = wei @ v

    return out



In [93]:
# Multi_layer attention

class MultiHeadAttention(nn.Module):

  def __init__(self , num_head , head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_head)])


  def forward(self , x):
    return torch.cat([h(x) for h in self.heads], dim =-1 )


In [94]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
vocab_size = 65
n_embd = 32

class BigramLanguageModel(nn.Module):

  def __init__(self ,vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size ,n_embd) #[65 , 32]
    self.position_embedding_table = nn.Embedding(BLOCK_SIZE, n_embd) #[8 ,32]
    self.sa_heads = MultiHeadAttention( 4 , n_embd//4)
    self.lm_head = nn.Linear(n_embd , vocab_size)


  def forward( self ,idx ,targets =None):
    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx) # (B=batch[32], T= token/block_size[8], C= channels[32] )


    pos_emb = self.position_embedding_table(torch.arange(T , device = device))  # [T , C]


    x = tok_emb + pos_emb # [B , T , C]
    x = self.sa_heads(x)
    logits = self.lm_head(x)  #[B ,T , vocab_size]

    if targets is None:
     loss =None

    else:
      # Reshape logits and targets for F.cross_entropy
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits , targets)

    return logits , loss

  def generate(self , idx , max_new_token): # there idx is shape is (32,8) batch=32 , block/token=8
    for _ in range(max_new_token):

     idx_cond = idx[: ,-BLOCK_SIZE:]  # logits initailly is (32,8,65) we are taking all batch(32) last token/letter and its all 65 next charecters possiblity now shape is(32 ,65)
     logits , loss = self(idx_cond)
     logits = logits[:, -1 ,:] # doing forword pass
     probs = F.softmax(logits ,dim =-1) #(32,65)
     idx_next = torch.multinomial(probs , num_samples=1) #(4,1)
     idx = torch.cat((idx , idx_next) , dim =1) #(4 ,9)

    return idx


model_2 = BigramLanguageModel(vocab_size)
logits , loss= model_2(xb , yb)


print(logits.shape)
print(loss)

torch.Size([256, 65])
tensor(4.2172, grad_fn=<NllLossBackward0>)


In [95]:
optimizer = torch.optim.AdamW(params = model_2.parameters(), lr =1e-3)

In [96]:

model_2 = model_2.to(device)
for i in range(5000):

  xb , yb = get_batch(split = 'train', block_size = 8 , batch_size = 32)
  xb , yb =xb.to(device) ,yb.to(device)
  logits , loss = model_1( xb , yb)
  optimizer.zero_grad(set_to_none = True)
  loss.backward()
  optimizer.step()

print(loss.item())
idx = torch.zeros((1,1), dtype = torch.long)
idx = idx.to(device)
print(decode(model_2.generate(idx , max_new_token= 300)[0].tolist()))

3.7137622833251953

a;HQPkMUwrDybSyViDgCw3RsVwyDh.qHWFrsgJqi$BMEQoDtVuiU-mVOipxBlEo
RpBCOQJ3r$V,:yvtn&q;;AVY,p

$QUaB,lE?,DuF uoVqHarAcqDtcq
qcx
&vb3I,Eas;nPH' K$g,CnwF?IiiRXr;V,x E;e$,POgxepiPTEJO-jn&Qge'BwKFeT?hkfc,bXxUJ!GSKYM.gZCwo&f-;LiBbkhgwEJC lHYzqdfyfY.kaAbtRasVqmdPWcRjUmRaLJRZ Rsacn3TaA.JsD'UrN.XavW,r,lLtXjz-C
